# Phase 0: Baseline Reproduction & Hidden State Inspection

**Goals:**
1. Install beat_this and verify it runs on Colab T4
2. Inspect the model architecture to find hidden state hook points
3. Run baseline inference on a test set
4. Compute standard MIR metrics and compare to published numbers

**Runtime:** Set to GPU (Runtime > Change runtime type > T4 GPU)

In [ ]:
# Install dependencies
!pip install -q git+https://github.com/CPJKU/beat_this.git mir_eval librosa soundfile mirdata

In [ ]:
import torch
import numpy as np

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

## 1. Load beat_this and inspect architecture

This is the critical step: we need to find which layer outputs the hidden
states that our confidence/tempo heads will attach to.

In [ ]:
# Load the beat_this model
# The beat_this package provides a high-level inference API
# but we need to look under the hood at the model architecture

from beat_this.inference import File2Beats

# Load the model — this downloads weights automatically
model = File2Beats(checkpoint_path="final0", device="cuda" if torch.cuda.is_available() else "cpu")

print("Model loaded successfully")
print(f"Device: {model.device}")

In [ ]:
# Inspect the full model architecture
# This tells us every layer name and type — we need to find
# the transformer output BEFORE the final beat/downbeat projection

print("=" * 80)
print("FULL MODEL ARCHITECTURE")
print("=" * 80)

# Access the underlying PyTorch model
# The File2Beats wrapper has the actual nn.Module inside
net = model.model if hasattr(model, 'model') else model

for name, module in net.named_modules():
    n_params = sum(p.numel() for p in module.parameters(recurse=False))
    if n_params > 0:
        print(f"{name:60s} {type(module).__name__:30s} params={n_params:,}")

print("\n" + "=" * 80)
print("TOP-LEVEL CHILDREN")
print("=" * 80)
for name, child in net.named_children():
    n_params = sum(p.numel() for p in child.parameters())
    print(f"{name:40s} {type(child).__name__:30s} total_params={n_params:,}")

In [ ]:
# Count total parameters
total = sum(p.numel() for p in net.parameters())
trainable = sum(p.numel() for p in net.parameters() if p.requires_grad)
print(f"Total parameters: {total:,}")
print(f"Trainable: {trainable:,}")

In [ ]:
# Find the output projection layers (beat + downbeat heads)
# These are the layers AFTER the hidden states we want to tap

print("=" * 80)
print("LOOKING FOR OUTPUT PROJECTION LAYERS")
print("=" * 80)

for name, module in net.named_modules():
    if isinstance(module, torch.nn.Linear):
        print(f"{name:60s} in={module.in_features} out={module.out_features}")

In [ ]:
# Hook into the layer BEFORE the output projection to capture hidden states
# We'll identify the right layer from the architecture inspection above,
# then register a forward hook

hidden_states_captured = {}

def make_hook(name):
    def hook_fn(module, input, output):
        hidden_states_captured[name] = output.detach()
    return hook_fn

# Register hooks on candidate layers
# UPDATE THESE based on the architecture inspection output above
hooks = []
for name, module in net.named_modules():
    # Hook all layers that could be the pre-projection hidden state
    # We'll narrow this down after seeing the output shapes
    if any(keyword in name.lower() for keyword in ['norm', 'transformer', 'encoder', 'block']):
        if not any(keyword in name.lower() for keyword in ['embed', 'patch', 'pos']):
            h = module.register_forward_hook(make_hook(name))
            hooks.append(h)

print(f"Registered {len(hooks)} hooks")

In [ ]:
# Run a dummy forward pass to capture hidden states
# beat_this expects a mel spectrogram input

import librosa

# Generate a 10-second sine wave as test input
sr = 22050
duration = 10.0
t = np.linspace(0, duration, int(sr * duration))
# Mix of frequencies to simulate music-like content
audio = 0.5 * np.sin(2 * np.pi * 100 * t) + 0.3 * np.sin(2 * np.pi * 200 * t)
audio = audio.astype(np.float32)

# Use the model's built-in preprocessing if available,
# or compute mel spectrogram manually
print(f"Test audio: {duration}s, {sr}Hz, shape={audio.shape}")

# Try running inference to trigger hooks
try:
    # File2Beats may need a file path — save temp audio
    import soundfile as sf
    import tempfile
    with tempfile.NamedTemporaryFile(suffix='.wav', delete=False) as f:
        sf.write(f.name, audio, sr)
        temp_path = f.name
    
    beats, downbeats = model(temp_path)
    print(f"Inference succeeded: {len(beats)} beats, {len(downbeats)} downbeats")
except Exception as e:
    print(f"Direct inference failed: {e}")
    print("Trying manual forward pass...")

In [ ]:
# Inspect captured hidden states
print("=" * 80)
print("CAPTURED HIDDEN STATES")
print("=" * 80)

for name, tensor in hidden_states_captured.items():
    if isinstance(tensor, torch.Tensor):
        print(f"{name:60s} shape={list(tensor.shape)}")
    elif isinstance(tensor, tuple):
        shapes = [list(t.shape) if isinstance(t, torch.Tensor) else type(t).__name__ for t in tensor]
        print(f"{name:60s} tuple: {shapes}")

# Clean up hooks
for h in hooks:
    h.remove()

print("\n>>> LOOK FOR: a [B, T, D] tensor where D is the hidden dim (likely 256 or 512)")
print(">>> That tensor is what our confidence/tempo heads will train on")

## 2. Baseline evaluation on Ballroom dataset

Now run beat_this on a standard benchmark and compute metrics.
We use Ballroom (698 tracks) via mirdata for evaluation only (CC BY-NC-SA).

In [ ]:
# Download Ballroom dataset annotations via mirdata
import mirdata

# mirdata handles annotation download; audio must be obtained separately
ballroom = mirdata.initialize('ballroom', data_home='/content/mirdata')
ballroom.download()  # downloads annotations only

track_ids = ballroom.track_ids
print(f"Ballroom tracks: {len(track_ids)}")

In [ ]:
# Check what's available — we need audio files for inference
# mirdata may or may not provide audio download depending on the dataset

sample_track = ballroom.track(track_ids[0])
print(f"Sample track: {track_ids[0]}")
print(f"Audio path: {sample_track.audio_path}")
print(f"Audio exists: {sample_track.audio_path and __import__('os').path.exists(sample_track.audio_path)}")

if sample_track.beats is not None:
    print(f"Beat annotations: {len(sample_track.beats.times)} beats")
    print(f"First 5 beat times: {sample_track.beats.times[:5]}")
    print(f"First 5 beat positions: {sample_track.beats.positions[:5]}")

In [ ]:
# If audio is available, run evaluation
# Otherwise, use beat_this's own evaluation script as reference

import mir_eval
import os
from collections import defaultdict

results = defaultdict(list)
n_evaluated = 0
n_skipped = 0

for tid in track_ids:
    track = ballroom.track(tid)
    
    # Skip if no audio
    if not track.audio_path or not os.path.exists(track.audio_path):
        n_skipped += 1
        continue
    
    # Skip if no beat annotations
    if track.beats is None:
        n_skipped += 1
        continue
    
    try:
        # Run beat_this inference
        pred_beats, pred_downbeats = model(track.audio_path)
        
        # Get reference annotations
        ref_beats = track.beats.times
        
        # Compute metrics
        scores = mir_eval.beat.evaluate(ref_beats, pred_beats)
        for metric, value in scores.items():
            results[metric].append(value)
        
        n_evaluated += 1
        
        if n_evaluated % 50 == 0:
            print(f"Evaluated {n_evaluated} tracks...")
    
    except Exception as e:
        print(f"Error on {tid}: {e}")
        n_skipped += 1

print(f"\nEvaluated: {n_evaluated}, Skipped: {n_skipped}")

In [ ]:
# Print results
if results:
    print("=" * 60)
    print("BALLROOM BASELINE RESULTS")
    print("=" * 60)
    for metric in ['F-measure', 'Cemgil', 'Goto', 'CMLt', 'CMLc', 'AMLt', 'AMLc']:
        if metric in results:
            vals = results[metric]
            print(f"{metric:15s}: {np.mean(vals):.4f} (+/- {np.std(vals):.4f})")
    
    print("\n>>> Compare these to beat_this published numbers")
    print(">>> Phase 0 pass criterion: within 1% of published F-measure")
else:
    print("No results — audio files likely not available via mirdata.")
    print("Options:")
    print("  1. Download Ballroom audio separately and point mirdata to it")
    print("  2. Use beat_this's own eval script: https://github.com/CPJKU/beat_this")
    print("  3. Test on your own audio files instead")

## 3. Summary

After running this notebook, you should know:

1. **The exact layer name** for the hidden state hook in `model.py`
2. **The hidden dimension** (D) for head initialization
3. **Baseline metric numbers** to compare against published results

### Next steps
- Update `src/beat_grid_confidence/model.py` with the discovered hook point
- Update `configs/base.yaml` with the correct `hidden_dim`
- If baseline matches published numbers: Phase 0 pass, move to Phase 1
- Create notebook `02_confidence_head_training.ipynb` for Phase 1